# Continuum | Offline customer intelligence model
UCI Online Retail II (CC BY 4.0). Historical UK retail transactions in GBP, 2009–2011. Churn is operationally defined as no positive purchase in the next 90 days, not a confirmed account cancellation. Use chronological, outcome-window-separated splits. No training happens in the hosted app.


In [1]:
from pathlib import Path
import io, zipfile, urllib.request, json, hashlib
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_score, recall_score, f1_score, mean_absolute_error
from sklearn.cluster import KMeans
ROOT=Path.cwd()
if not (ROOT/'static').exists(): ROOT=ROOT/'continuum'
OUT=ROOT/'models'; OUT.mkdir(exist_ok=True)
url='https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'
with urllib.request.urlopen(url, timeout=180) as resp: archive=resp.read()
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    book=next(n for n in z.namelist() if n.lower().endswith('.xlsx'))
    raw=pd.concat(pd.read_excel(io.BytesIO(z.read(book)),sheet_name=None,engine='openpyxl').values(),ignore_index=True)
raw.columns=[str(c).strip().lower().replace(' ','') for c in raw.columns]
raw=raw.rename(columns={'customerid':'customer','invoicedate':'date','invoice':'invoice','price':'price','quantity':'qty'})
raw['date']=pd.to_datetime(raw['date'])
raw['customer']=pd.to_numeric(raw['customer'],errors='coerce')
valid=raw.dropna(subset=['customer','date','invoice','price','qty']).copy()
valid=valid[(valid.qty>0)&(valid.price>0)&~valid.invoice.astype(str).str.startswith('C')]
valid['customer']=valid.customer.astype(int)
valid['revenue']=valid.qty*valid.price
orders=valid.groupby(['customer','invoice'],as_index=False).agg(date=('date','min'),revenue=('revenue','sum'))
print({'raw_lines':len(raw),'positive_purchase_lines':len(valid),'orders':len(orders),'first':str(orders.date.min()),'last':str(orders.date.max())})
del raw,valid,archive



{'raw_lines': 1067371, 'positive_purchase_lines': 805549, 'orders': 36969, 'first': '2009-12-01 07:45:00', 'last': '2011-12-09 12:50:00'}


## Point-in-time features and outcome construction


In [2]:
FEATURES=['recency','tenure','orders_30','orders_90','orders_180','spend_30','spend_90','spend_180','avg_order_180']
def snapshot(day, label=True):
    cut=pd.Timestamp(day); past=orders[orders.date<cut]
    recent=past[past.date>=cut-pd.Timedelta(days=180)]
    g=recent.groupby('customer')
    x=g.agg(last=('date','max'),orders_180=('invoice','size'),spend_180=('revenue','sum'))
    x['recency']=(cut-x['last']).dt.total_seconds()/86400
    x['tenure']=(cut-past.groupby('customer').date.min()).dt.total_seconds()/86400
    for n in (30,90):
        t=recent[recent.date>=cut-pd.Timedelta(days=n)].groupby('customer').agg(o=('invoice','size'),s=('revenue','sum'))
        x[f'orders_{n}']=t.o.reindex(x.index,fill_value=0)
        x[f'spend_{n}']=t.s.reindex(x.index,fill_value=0)
    x['avg_order_180']=x.spend_180/x.orders_180
    x=x[FEATURES].copy()
    if label:
        assert cut+pd.Timedelta(days=90)<=orders.date.max()
        future=orders[(orders.date>=cut)&(orders.date<cut+pd.Timedelta(days=90))].groupby('customer').revenue.sum()
        x['future_revenue']=future.reindex(x.index,fill_value=0)
        x['churn']=(x.future_revenue==0).astype(int)
    x['as_of']=str(cut.date())
    return x.reset_index()
train_dates=pd.date_range('2010-06-01','2010-10-01',freq='MS')
train=pd.concat([snapshot(d) for d in train_dates],ignore_index=True)
val=snapshot('2011-01-01')
test=snapshot('2011-05-01')
assert pd.Timestamp(train.as_of.max())+pd.Timedelta(days=90)<=pd.Timestamp(val.as_of.min())
assert pd.Timestamp(val.as_of.max())+pd.Timedelta(days=90)<=pd.Timestamp(test.as_of.min())
print([(len(d),round(d.churn.mean(),3)) for d in [train,val,test]])



[(13989, np.float64(0.444)), (3415, np.float64(0.623)), (3049, np.float64(0.521))]


## Candidate selection, untouched temporal test and separate revenue model


In [3]:
def X(d): return d[FEATURES].astype(float)
candidates={
 'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42)),
 'Random forest':RandomForestClassifier(n_estimators=160,max_depth=9,min_samples_leaf=12,n_jobs=2,random_state=42),
 'Extra trees':ExtraTreesClassifier(n_estimators=160,max_depth=10,min_samples_leaf=12,n_jobs=2,random_state=42),
 'Histogram gradient boosting':HistGradientBoostingClassifier(max_iter=180,max_leaf_nodes=15,l2_regularization=3,random_state=42)}
scores={}
for name,m in candidates.items():
    m.fit(X(train),train.churn)
    p=m.predict_proba(X(val))[:,1]
    scores[name]={'pr_auc':float(average_precision_score(val.churn,p)), 'brier':float(brier_score_loss(val.churn,p))}
winner=max(scores,key=lambda k:scores[k]['pr_auc'])
model=candidates[winner]
# Fit a probability calibrator using the validation snapshot only.
def logit(p): return np.log(np.clip(p,1e-6,1-1e-6)/np.clip(1-p,1e-6,1)).reshape(-1,1)
cal=LogisticRegression(C=1e3).fit(logit(model.predict_proba(X(val))[:,1]),val.churn)
prob=lambda d:cal.predict_proba(logit(model.predict_proba(X(d))[:,1]))[:,1]
p=prob(test)
metrics={'pr_auc':float(average_precision_score(test.churn,p)),'roc_auc':float(roc_auc_score(test.churn,p)),'brier':float(brier_score_loss(test.churn,p)),'precision':float(precision_score(test.churn,p>=.5,zero_division=0)),'recall':float(recall_score(test.churn,p>=.5,zero_division=0)),'f1':float(f1_score(test.churn,p>=.5,zero_division=0)),'prevalence':float(test.churn.mean()),'baseline_pr_auc':float(test.churn.mean()),'baseline_brier':float(brier_score_loss(test.churn,np.full(len(test),train.churn.mean())))}
active=train[train.churn==0]
rev=HistGradientBoostingRegressor(loss='squared_error',max_iter=150,max_leaf_nodes=15,l2_regularization=5,random_state=42).fit(X(active),np.log1p(active.future_revenue))
conditional=lambda d:np.maximum(0,np.expm1(rev.predict(X(d))))
metrics['future_revenue_mae_gbp']=float(mean_absolute_error(test.future_revenue,(1-p)*conditional(test)))
metrics['future_revenue_baseline_mae_gbp']=float(mean_absolute_error(test.future_revenue,test.spend_90))
# Segmentation uses only training features.
seg=make_pipeline(StandardScaler(),KMeans(n_clusters=4,n_init=10,random_state=42)).fit(np.log1p(X(train)))
print(winner,scores,metrics)



/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Extra trees {'Logistic regression': {'pr_auc': 0.8275903889475602, 'brier': 0.2611960378111868}, 'Random forest': {'pr_auc': 0.8269293360971327, 'brier': 0.24994004415449822}, 'Extra trees': {'pr_auc': 0.8310152444228796, 'brier': 0.2455298107123302}, 'Histogram gradient boosting': {'pr_auc': 0.8293644100540609, 'brier': 0.2596532224686266}} {'pr_auc': 0.7471059490176002, 'roc_auc': 0.7577937370801201, 'brier': 0.21271796567732268, 'precision': 0.639212175470009, 'recall': 0.8992443324937027, 'f1': 0.7472527472527473, 'prevalence': 0.5208265004919646, 'baseline_pr_auc': 0.5208265004919646, 'baseline_brier': 0.25541504485124317, 'future_revenue_mae_gbp': 422.0860225486561, 'future_revenue_baseline_mae_gbp': 383.4952020334536}


## Export immutable inference artifacts; no model training in serving process


In [4]:
demo=snapshot('2011-09-01',label=False)
demo['customer']=demo.customer.map(lambda v:'C-'+hashlib.sha256(('continuum-demo-'+str(v)).encode()).hexdigest()[:10])
demo=demo.rename(columns={'customer':'customer_id'})
demo[['customer_id']+FEATURES].to_csv(OUT/'demo.csv',index=False,float_format='%.4f')
bundle={'model':model,'calibrator':cal,'revenue':rev,'segment':seg,'features':FEATURES,'medians':X(train).median().to_dict()}
joblib.dump(bundle,OUT/'bundle.joblib',compress=3)
report={'selected_model':winner,'validation':scores,'test':metrics,'train_rows':len(train),'validation_rows':len(val),'test_rows':len(test),'train_dates':[str(d.date()) for d in train_dates],'validation_date':'2011-01-01','test_date':'2011-05-01','demo_date':'2011-09-01','demo_rows':len(demo),'dataset_url':'https://archive.ics.uci.edu/dataset/502/online+retail+ii','license':'CC BY 4.0','target':'No positive purchase in following 90 days','currency':'GBP','limitations':'Historical single retailer; repeat customers can appear in different time snapshots; no randomized campaign data; conditional revenue is a log-scale model estimate, not audited cash flow.'}
(OUT/'report.json').write_text(json.dumps(report,indent=2))
display(pd.DataFrame(scores).T)
display(pd.Series(metrics))



,pr_auc,brier
Logistic regression,0.827590,0.261196
Random forest,0.826929,0.249940
Extra trees,0.831015,0.245530
Histogram gradient boosting,0.829364,0.259653


pr_auc                               0.747106
roc_auc                              0.757794
brier                                0.212718
precision                            0.639212
recall                               0.899244
f1                                   0.747253
prevalence                           0.520827
baseline_pr_auc                      0.520827
baseline_brier                       0.255415
future_revenue_mae_gbp             422.086023
future_revenue_baseline_mae_gbp    383.495202
dtype: float64